# RQ1.1 — Temporal behavior of the token embeddings

Worked example: **FID 306 (clear cut vegetation), tile 0, cloud filter v3**.

Chosen because it is the candidate with the best joint coverage after the event
(optical 5/3/4, joint 5/1/2 over the before, event and after windows), it is
covered by a single tile, so tile and polygon coincide and the block artifact of
Section 4.1.1 does not apply, and it is one of the 346 polygons of the
quantitative analysis, so the example is traceable across research questions.

The three readings of the methodology, each for the optical, SAR and joint modes:

1. **Similarity between dates** — the $T \times T$ cosine similarity matrix
2. **Most variable dimension** — the single embedding coordinate that responds most
3. **PCA trajectory** — PC1 to PC3 projected on axes fitted once on a reference image

Every figure is exported to PDF with the same publication settings as
`croma_fid_embeddings_charlotte.ipynb`.

In [ ]:
# ── 0. working directory ────────────────────────────────────────────────
# All paths in time_series_pipeline are relative (embeddings/, data_csv/,
# data_shp/), so the notebook MUST run from the repository root. Without this
# the globs return empty and every function skips silently.
import os

REPO = os.path.expanduser("~/tropical_forest_disturbance")
os.chdir(REPO)
for d in ("embeddings", "data_csv", "data_shp"):
    assert os.path.isdir(d), f"{d} not found — wrong cwd: {os.getcwd()}"
print("cwd:", os.getcwd())

In [ ]:
# ── 1. publication figure config ────────────────────────────────────────
# Same settings as croma_fid_embeddings_charlotte.ipynb, cell 50.
from pathlib import Path
import matplotlib.pyplot as plt

try:
    import scienceplots  # noqa: F401
    plt.style.use(["science", "no-latex"])
except (ImportError, OSError):
    print("scienceplots not available, falling back to the default style")

TEXT_W = 160 / 25.4           # 6.30 in, text width of the document
FIGDIR = Path("figures_rq1")  # copy down to the manuscript figures/ afterwards
FIGDIR.mkdir(exist_ok=True)

plt.rcParams.update({
    "font.size": 12, "axes.labelsize": 12, "axes.titlesize": 11,
    "xtick.labelsize": 11, "ytick.labelsize": 11, "legend.fontsize": 10,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.5,
    "savefig.bbox": "tight",
})
print("figures ->", FIGDIR.resolve())

In [ ]:
# ── 2. capture the figures the pipeline draws ───────────────────────────
# The plotting helpers call plt.show() internally and neither return nor save
# their figures, so plt.show is wrapped to export each one to PDF at the
# document text width before displaying it.
#
# time_series_pipeline calls plt.show() ten times and plt.close() never, so if
# the backend does not close on show the figures pile up and every later call
# re-saves all of them. Hence the explicit close at the end.
_orig_show = plt.show
_state = {"prefix": "fig", "n": 0}


def set_prefix(prefix):
    """Name the PDFs of everything drawn from here on."""
    _state["prefix"], _state["n"] = prefix, 0


def _fit_width(fig, width=TEXT_W):
    """Scale to the text width, keeping the aspect ratio."""
    w, h = fig.get_size_inches()
    fig.set_size_inches(width, h * width / w)


def _save_and_show(*args, **kwargs):
    nums = plt.get_fignums()
    for num in nums:
        fig = plt.figure(num)
        _state["n"] += 1
        out = FIGDIR / f"{_state['prefix']}_{_state['n']:02d}.pdf"
        _fit_width(fig)
        fig.savefig(out)
        print(f"    saved {out.name}")
    _orig_show(*args, **kwargs)
    for num in nums:
        plt.close(num)


plt.show = _save_and_show


def clear_figures():
    """Wipe FIGDIR. Run before a clean pass so old PDFs do not linger."""
    n = 0
    for p in FIGDIR.glob("*.pdf"):
        p.unlink(); n += 1
    print(f"removed {n} pdf")


print("plt.show patched")

In [ ]:
# ── 3. the worked example ───────────────────────────────────────────────
import time_series_pipeline as tsp
from time_series_pipeline import (CSV_TEMPLATE, _create_embeddings,
                                  _allowed_ids_for_fid, _cosine_sim_plot,
                                  _most_variable_dim, _pca_first_image,
                                  _pick_random_token, _show_tile_with_pixel,
                                  _show_explicit_token, _event_date,
                                  _paths_for, _by_win)
from collections import Counter

FID, TILE, VERSION = 306, 0, "v3"
TOK_R, TOK_C = None, None    # None -> random token inside the polygon, seed 42
SEED = 42
MODES = ("optical", "sar", "joint")

# _VERSION is a module global that appears in every plot title. The helpers
# below do not set it (only complete_analysis does), so set it here.
tsp._VERSION = VERSION

csv_path = CSV_TEMPLATE.format(version=VERSION)
_create_embeddings(FID, csv_path, 7)          # skips whatever already exists
allowed = _allowed_ids_for_fid(csv_path, FID)
S2, S1 = allowed["s2"], allowed["s1"]

print(f"fid {FID} tile {TILE} — {VERSION}")
print(f"  event date: {_event_date(FID)}")
for mode in MODES:
    c = Counter(_by_win(p) for p in _paths_for(FID, TILE, mode, S2, S1))
    print(f"  {mode:<8} bef={c['bef']:>3} evt={c['evt']:>3} aft={c['aft']:>3}")

In [ ]:
# pick (or show) the token, and see where it falls inside the polygon
clear_figures()
set_prefix(f"rq11_fid{FID}_token")
if TOK_R is None or TOK_C is None:
    TOK_R, TOK_C, pix_row, pix_col, rgb_tile, poly_pix = _pick_random_token(
        FID, TILE, SEED)
    _show_tile_with_pixel(FID, TILE, pix_row, pix_col, rgb_tile, poly_pix)
    print(f"  random token -> ({TOK_R}, {TOK_C})   "
          f"# write these into TOK_R / TOK_C above to freeze the figures")
else:
    _show_explicit_token(FID, TILE, TOK_R, TOK_C)
    print(f"  explicit token ({TOK_R}, {TOK_C})")

## Reading 1 — similarity between dates

One $T \times T$ cosine similarity matrix per mode. Dates whose embeddings are
alike form blocks, so a disturbance should appear as a boundary between blocks at
the event date.

This is the reading that carries the **modality** axis of RQ1.1.

In [ ]:
for mode in MODES:
    print(f"### cosine {mode}")
    set_prefix(f"rq11_fid{FID}_cosine_{mode}")
    _cosine_sim_plot(FID, TILE, TOK_R, TOK_C, mode, S2, S1)

## Reading 2 — most variable dimension

The tokens of the tile are averaged at each date, giving an array of dates by 768
dimensions; the dimension with the largest standard deviation across the series
is followed through time.

The dimension selected depends on the tile and on the mode, so these panels are
**not comparable with one another** — the quantity plotted is a different
coordinate in each. One panel is enough for the manuscript; the three are run
here only to see which mode responds.

In [ ]:
for mode in MODES:
    print(f"### most variable dim {mode}")
    set_prefix(f"rq11_fid{FID}_mvd_{mode}")
    _most_variable_dim(FID, TILE, mode, S2, S1)

## Reading 3 — PCA trajectory, PC1 to PC3

PCA of three components, fitted once on a single reference image (the earliest
Sentinel-2 date common to the three cloud filters) using its 225 tokens as
samples. The token series is projected onto those fixed axes.

`_pca_first_image` draws two figures per mode: `_01` is PC1 alone, `_02` is
**PC1 to PC3 together with the variance each explains** — that second one is the
figure for the manuscript.

The axes are fitted per mode, so PC1 in optical and PC1 in joint are different
axes: only the shape of the trajectory is comparable between modes, never the
values. State this in the caption.

In [ ]:
for mode in MODES:
    print(f"### PCA {mode}")
    set_prefix(f"rq11_fid{FID}_pca_{mode}")
    _pca_first_image(FID, TILE, TOK_R, TOK_C, mode, S2, S1)

## RQ1.2 — cloud filter comparison

The same token read under v1, v2 and v3, so the only quantity differing between
the panels is which dates survive. One figure per mode, three columns (the
filters) by three rows (most variable dimension, PC1, PC1 to PC3).

The methodology reports the PCA rows. Note this figure is born at 18 by 12
inches; scaled to the text width it may be too dense to read, in which case
export it separately at full page width or one mode at a time.

In [ ]:
from time_series_pipeline import compare_mvd

set_prefix(f"rq12_filters_fid{FID}")
compare_mvd(FID, tok_r=TOK_R, tok_c=TOK_C, tile=TILE, modalities=MODES)

## Notes

- **Freeze the token.** Run once with `TOK_R = TOK_C = None`, read the coordinates
  printed above, write them into the definition cell and re run. Otherwise the
  figures depend on the seed.
- **Watch the joint panels.** Under v3 this polygon has two dates in the after
  window in joint mode, against four in optical, and that is the best joint
  coverage among the candidates. With two points after the event the joint
  trajectory supports a weaker claim than the optical one. Say so in the text
  rather than reading it as if it were the same evidence.
- **RQ1.1 also asks about disturbance types.** This notebook covers the modality
  axis on one polygon. The type axis needs one more figure: PC1 for one polygon
  per class, six small panels. Without it, half of RQ1.1 goes unanswered.
- Copy the PDFs to `manuscript_tropical_forest_disturbance/figures/` and rename
  the ones that make it into the chapter.